In [ ]:
import importlib
import inspect
import sys
import zipfile
from pathlib import Path
import torch

# 1. Check T4 GPU and compressed file
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled on Colab! Please select Runtime -> Change runtime type -> T4 GPU.")

zip_path = Path("retrieval_eval_colab.zip")
if not zip_path.exists() or not zipfile.is_zipfile(zip_path):
    raise RuntimeError("Valid 'retrieval_eval_colab.zip' file not found. Please wait for 100% upload completion!")

print(f">>> GPU detected: {torch.cuda.get_device_name(0)}. Installing libraries and extracting...")
!pip install -q chromadb rank-bm25 pyyaml langchain-community > /dev/null 2>&1
!rm -rf "Evaluation" "Retrieval" "Query Understanding and Routing" "Document Ingestion and Parsing" path.py retrieval_eval_results.zip
!unzip -q -o retrieval_eval_colab.zip

# 2. Configure paths and reload clean modules
ROOT_DIR = Path.cwd()
for p in (
    ROOT_DIR,
    ROOT_DIR / "Retrieval",
    ROOT_DIR / "Query Understanding and Routing",
    ROOT_DIR / "Document Ingestion and Parsing" / "Metadata",
    ROOT_DIR / "Document Ingestion and Parsing" / "Provenance and Traceability",
):
    if p.is_dir() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from sentence_transformers import CrossEncoder
import query_vectorizer
import reranker
import retrieval_system

importlib.reload(query_vectorizer)
importlib.reload(reranker)
importlib.reload(retrieval_system)

from query_vectorizer import QueryVectorizer
from reranker import CrossEncoderReranker

# 3. CrossEncoder.predict compatibility patch on Colab + FP16 acceleration on T4 GPU
def _safe_ce_predict(self, sentences, batch_size: int = 32, show_progress_bar: bool = False, **kwargs):
    if not sentences:
        return []
    dev = next(self.model.parameters()).device
    max_len = getattr(self, "max_seq_length", None) or getattr(self, "max_length", None) or 512
    all_scores: list[float] = []
    self.model.eval()
    with torch.inference_mode():
        for i in range(0, len(sentences), batch_size):
            batch = sentences[i : i + batch_size]
            enc = self.tokenizer(
                [str(p[0]) for p in batch],
                [str(p[1]) for p in batch],
                padding=True,
                truncation=True,
                max_length=max_len,
                return_tensors="pt",
            )
            enc = {k: v.to(dev) for k, v in enc.items()}
            logits = self.model(**enc).logits.view(-1)
            all_scores.extend(torch.sigmoid(logits).float().cpu().tolist())
    return all_scores

CrossEncoder.predict = _safe_ce_predict

_orig_qv_get = QueryVectorizer._get_model
def _cuda_qv_get(self):
    self.device = "cuda"
    m = _orig_qv_get(self).to("cuda")
    if not getattr(m, "_is_fp16", False):
        m.half()
        m._is_fp16 = True
    return m

_orig_ce_get = CrossEncoderReranker._get_model
def _cuda_ce_get(self):
    self.device = "cuda"
    ce = _orig_ce_get(self)
    ce.to("cuda")
    if hasattr(ce, "model") and not getattr(ce.model, "_is_fp16", False):
        ce.model.half()
        ce.model._is_fp16 = True
    return ce

QueryVectorizer._get_model = _cuda_qv_get
CrossEncoderReranker._get_model = _cuda_ce_get

# 4. Run 125-sample Retrieval evaluation script
eval_script = Path("Evaluation/Retrieval Evals/eval_retrieval.py")
code = eval_script.read_text(encoding="utf-8")
sys.argv = [str(eval_script), "--strategy", "semantic", "--device", "cuda"]
exec(compile(code, str(eval_script), "exec"), {"__name__": "__main__", "__file__": str(eval_script.resolve())})

# 5. Compress JSON report and download to Laptop
!zip -q -j retrieval_eval_results.zip "Evaluation/Retrieval Evals/retrieval_eval_report.json"
from google.colab import files
files.download("retrieval_eval_results.zip")

>>> GPU nhận diện: Tesla T4. Đang cài thư viện và giải nén...
>>> Khởi tạo hệ thống đánh giá Retrieval (strategy='semantic', device='cuda', top_k=5)...
>>> Đang chạy warm-up model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

... (23 outputs left out)

   ... Đã xử lý 75/125 mẫu (82.7s)
   ... Đã xử lý 100/125 mẫu (101.5s)


   ... Đã xử lý 125/125 mẫu (109.1s)

KẾT QUẢ ĐÁNH GIÁ RETRIEVAL PIPELINE (FIXED PIPELINE @ TOP-5, N = 125)

### Bảng 1. Tổng hợp Chất lượng Truy xuất (100 Mẫu Positive)
| Token Recall@5 | Token Precision@5 | Token IoU@5 | MRR@5 | nDCG@5 | ILS@5 | Unique Ev. Cov@5 |
| :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| **66.41%** | **19.72%** | **18.3%** | **0.7023** | **0.5782** | **0.6891** | **66.0%** |

### Bảng 2. Phân rã theo Nhóm Kịch bản (Categories)
| Category | Recall@5 | Precision@5 | IoU@5 | MRR@5 | nDCG@5 | ILS@5 | UEC@5 |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: |
| `single_aspect` | 78.93% | 22.14% | 22.13% | 0.6767 | 0.6145 | 0.7178 | 78.0% |
| `multi_evidence` | 50.09% | 28.56% | 25.46% | 0.7333 | 0.5043 | 0.6963 | 50.67% |
| `multi_document` | 36.6% | 14.63% | 12.06% | 0.486 | 0.3315 | 0.6738 | 35.33% |
| `hard_scope` | 100.0% | 13.55% | 13.55% | 0.9133 | 0.8623 | 0.6754 | 100.0% |

### Bảng 3. Phân rã theo Ngôn ngữ Truy vấn (Language)
| La

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>